```python
import json
from typing import List, Dict, Any

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# --- Constants ---
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_TOKENS = 100
TEMPERATURE = 0.0


# --- Tool Implementation ---
def get_current_wind_speed(location: str) -> float:
    """
    Get the current wind speed in km/h at a given location.

    Args:
        location: The location to get the temperature for, in the format "City, Country".
    Returns:
        The current wind speed at the given location in km/h, as a float.
    """
    print(f"Executing tool 'get_current_wind_speed' for location: {location}")
    return 6.0


# --- Core Logic Functions ---
def setup_model_and_tokenizer(model_id: str) -> (LLM, AutoTokenizer):
    """Initializes and returns the VLLM model and tokenizer."""
    print("Setting up model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    llm = LLM(model=model_id)
    print("Setup complete.")
    return llm, tokenizer


def complete_tool_call(
    llm: LLM,
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, Any]],
    partial_tool_call_str: str,
) -> str:
    """
    Generates the completion for a partial tool call string.
    """
    prompt_base = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        tools=[get_current_wind_speed],
        add_generation_prompt=True,
    )
    prompt = prompt_base + partial_tool_call_str

    print("--- Step 1: Prompt for Tool Call Completion ---")
    print(prompt)
    print("---------------------------------------------")

    sampling_params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    outputs = llm.generate([prompt], sampling_params)
    generated_suffix = outputs[0].outputs[0].text

    full_tool_call_str = partial_tool_call_str + generated_suffix
    print(f"\n--- Step 1: Completed Tool Call (Raw) ---\n{full_tool_call_str}")
    print("-----------------------------------------")

    return full_tool_call_str


def parse_and_execute_tool(full_tool_call_str: str) -> (str, dict, str):
    """
    Parses the tool call string, executes the function, and returns results.
    """
    try:
        first_brace = full_tool_call_str.find("{")
        last_brace = full_tool_call_str.rfind("}")
        if first_brace == -1 or last_brace == -1:
            raise ValueError("Could not find JSON object in the tool call string.")

        cleaned_json_str = full_tool_call_str[first_brace : last_brace + 1]
        tool_call_data = json.loads(cleaned_json_str)
        function_name = tool_call_data.get("name")

        if function_name != "get_current_wind_speed":
            raise ValueError(f"Unrecognized function name '{function_name}'.")

        parameters = tool_call_data.get("parameters", {})
        location = parameters.get("location")
        if not location:
            raise ValueError("'location' parameter not found in tool call")

        result = get_current_wind_speed(location=location)
        print(f"\n--- Function execution result: {result} ---")
        return function_name, parameters, str(result)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Failed to decode JSON from tool call: {e}. String was: '{cleaned_json_str}'"
        )


def generate_final_response(
    llm: LLM,
    tokenizer: AutoTokenizer,
    messages: List[Dict[str, Any]],
    function_name: str,
    parameters: dict,
    function_result: str,
) -> str:
    """
    Generates the final natural language response from the model.
    """
    conversation_history = messages.copy()
    tool_call_id = "call_weather_123"

    conversation_history.append(
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": tool_call_id,
                    "type": "function",
                    "function": {
                        "name": function_name,
                        "arguments": json.dumps(parameters),
                    },
                }
            ],
        }
    )

    conversation_history.append(
        {"role": "tool", "tool_call_id": tool_call_id, "content": function_result}
    )

    final_prompt = tokenizer.apply_chat_template(
        conversation_history, tokenize=False, add_generation_prompt=True
    )

    print("\n--- Step 2: Prompt for Final Response ---")
    print(final_prompt)
    print("-----------------------------------------")

    sampling_params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    final_outputs = llm.generate([final_prompt], sampling_params)
    final_response = final_outputs[0].outputs[0].text

    print("\n--- Step 2: Final Model Response ---")
    print(final_response.strip())
    print("------------------------------------")

    return final_response.strip()


def main():
    """Main execution flow for demonstrating forced tool call completion."""
    try:
        # 1. Setup
        llm, tokenizer = setup_model_and_tokenizer(MODEL_ID)

        # 2. Initial conversation
        messages = [{"role": "user", "content": "What is the wind speed in Beijing?"}]

        # 3. Define the partial tool call to force completion
        partial_tool_call_str = '<tool_call>{"name": "get_current_wind_speed", "parameters": {"location": '

        # 4. Complete the tool call
        full_tool_call_str = complete_tool_call(
            llm, tokenizer, messages, partial_tool_call_str
        )

        # 5. Parse and execute the completed tool call
        function_name, parameters, function_result = parse_and_execute_tool(
            full_tool_call_str
        )

        # 6. Generate the final natural language response
        generate_final_response(
            llm, tokenizer, messages, function_name, parameters, function_result
        )

    except (ValueError, json.JSONDecodeError) as e:
        print(f"Error processing tool call: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    main()

```